# 04 — Parallel Map-Reduce with Send

This is your A2H pattern — 15 Glue jobs × 3 profiles × 5 regions running in parallel.

```
START → spawn_workers ──→ process_region(na) ─┐
                       ──→ process_region(eu) ─┤ reducer merges
                       ──→ process_region(fe) ─┘
                                               → aggregate → END
```

`Send` = dynamic edge — one per item, all run in parallel.

In [ ]:
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.constants import Send

REGIONS = ["na", "eu", "fe", "jp"]
ROW_COUNTS = {"na": 50_000, "eu": 32_000, "fe": 18_000, "jp": 11_000}

class OverallState(TypedDict):
    regions: list[str]
    date: str
    results: Annotated[list[dict], operator.add]  # reducer — fan-in accumulator

class RegionState(TypedDict):  # state passed to each worker
    region: str
    date: str

In [ ]:
def spawn_workers(state: OverallState) -> list[Send]:
    """Conditional edge that fans out — one Send per region."""
    return [
        Send("process_region", {"region": r, "date": state["date"]})
        for r in state["regions"]
    ]

def process_region(state: RegionState) -> dict:
    """Worker — runs in parallel for each region."""
    rows = ROW_COUNTS[state["region"]]
    print(f"  [process_region] {state['region']} — {rows:,} rows")
    return {"results": [{"region": state["region"], "rows": rows, "status": "ok"}]}

def aggregate(state: OverallState) -> dict:
    """Fan-in — reducer already merged all results."""
    total = sum(r["rows"] for r in state["results"])
    print(f"\n[aggregate] {len(state['results'])} regions | {total:,} total rows")
    return {}

In [ ]:
builder = StateGraph(OverallState)
builder.add_node("process_region", process_region)
builder.add_node("aggregate", aggregate)

builder.add_conditional_edges(START, spawn_workers)   # fan-out
builder.add_edge("process_region", "aggregate")       # fan-in (reducer handles merge)
builder.add_edge("aggregate", END)

graph = builder.compile()

print("=== Parallel regional processing ===")
result = graph.invoke({"regions": REGIONS, "date": "2024-01-15", "results": []})
print(f"\nFinal results: {result['results']}")

## Key points

- `Send(node_name, state)` — dynamic edge, creates one task per item
- `Annotated[list, operator.add]` — reducer that accumulates across parallel branches (required for fan-in)
- Without the reducer, parallel branches would overwrite each other's results
- LangGraph runs `Send` targets concurrently in the same step

## Connecting to your A2H experience

> *"In A2H I ran 15 PySpark Glue jobs in parallel — 3 classification profiles × 5 regions. In LangGraph that maps directly to Send — one Send per (profile, region) tuple, each running the same process_region node, results merged by reducer. The 87.5% execution time reduction came from this parallelization plus switching from full-refresh to incremental delta."*